# Cyber Attack Dataset v3: EDA and Baseline

One 53-minute lab capture (439,171 frames) with four attacks:

`Normal` · `ARP_Spoofing` · `ARP_Storm` · `PING_Flood` · `SYN_Flood`

Version 3 publishes the values in their original Wireshark units and in capture order. This notebook covers:

1. Schema and missing values
2. Class imbalance
3. When each attack ran
4. What each attack looks like in the packet headers
5. A baseline on the chronological split, with and without the columns that give the answer away

Nothing in this notebook is fitted on the test rows.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

ORDER = ["Normal", "ARP_Spoofing", "ARP_Storm", "PING_Flood", "SYN_Flood"]


def find_data_dir():
    for root in (Path("/kaggle/input"), Path("release_v3"), Path(".")):
        if root.exists():
            hits = sorted(root.rglob("network_attack_traffic.csv"))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("network_attack_traffic.csv not found")


DATA = find_data_dir()
full = pd.read_csv(DATA / "network_attack_traffic.csv")
if "split" not in full.columns:
    raise RuntimeError("This notebook needs version 3 of the dataset (the table has no 'split' column).")
train, test = full[full.split == "train"], full[full.split == "test"]
print(f"full={full.shape}  train={len(train):,}  test={len(test):,}")
full.head()

## 1) Schema and missing values

A field is empty when the frame does not have that protocol layer. Earlier versions filled these with the column mean; here they are left empty.

In [ ]:
schema = pd.DataFrame({
    "dtype": full.dtypes.astype(str),
    "missing": full.isna().sum(),
    "missing_pct": (full.isna().mean() * 100).round(2),
    "unique": full.nunique(),
})
schema

In [ ]:
full[["frame_time_delta", "tcp_seq", "tcp_hdr_len", "data_len"]].describe().T.round(4)

## 2) Class imbalance

Normal traffic is about 98% of the rows, so accuracy says very little. The scores below are macro-F1 and per-class recall.

In [ ]:
counts = pd.read_csv(DATA / "label_mapping.csv")
display(counts)

ax = counts.set_index("label_name")["count"].plot(kind="bar", logy=True, rot=20, title="Rows per class (log scale)")
ax.set_ylabel("rows")
plt.tight_layout()
plt.show()

## 3) When each attack ran

`frame_number` is the position in the capture. The attacks ran in separate stretches, so the frame number alone predicts the class. That is why it must stay out of the feature set.

In [ ]:
full["second"] = full["frame_time_delta"].cumsum()
attacks = full[full.label != 0]

fig, ax = plt.subplots(figsize=(10, 3))
for i, name in enumerate(ORDER[1:]):
    part = attacks[attacks.label_name == name]
    ax.scatter(part["second"], np.full(len(part), i), s=4, label=name)
ax.set_yticks(range(4))
ax.set_yticklabels(ORDER[1:])
ax.set_xlabel("seconds into the capture")
ax.set_title("Attack packets over time")
plt.tight_layout()
plt.show()

full.groupby("label_name")["second"].agg(["min", "max"]).round(1).reindex(ORDER)

## 4) What each attack looks like

Each attack is one header pattern: ARP replies, ARP requests, ICMP type 0, or TCP segments with only SYN set.

In [ ]:
summary = full.groupby("label_name").agg(
    rows=("label", "size"),
    arp_reply=("arp_opcode", lambda s: (s == "reply").mean()),
    arp_request=("arp_opcode", lambda s: (s == "request").mean()),
    icmp=("icmp_type_code", lambda s: s.notna().mean()),
    tcp=("tcp_hdr_len", lambda s: s.notna().mean()),
    syn=("tcp_flag_syn", "mean"),
    ack=("tcp_flag_ack", "mean"),
    median_data_len=("data_len", "median"),
    median_time_delta=("frame_time_delta", "median"),
).reindex(ORDER)
summary.round(4)

Normal traffic contains the same patterns in small numbers: ARP requests and replies, and SYN segments from ordinary connections. Those are the rows a header-only model has to tell apart from the attacks.

In [ ]:
normal = full[full.label == 0]
pd.Series({
    "ARP requests in Normal": int((normal.arp_opcode == "request").sum()),
    "ARP replies in Normal": int((normal.arp_opcode == "reply").sum()),
    "SYN segments in Normal": int((normal.tcp_flag_syn == 1).sum()),
    "ICMP frames in Normal": int(normal.icmp_type_code.notna().sum()),
})

## 5) Baseline on the chronological split

Two feature sets:

- **header_only**: what a single packet header says.
- **with_shortcuts**: adds `frame_number` and the device roles. These name the time of the attack and the attacker.

In [ ]:
FLAGS = ["tcp_flag_fin", "tcp_flag_syn", "tcp_flag_rst", "tcp_flag_psh", "tcp_flag_ack"]
HEADER = ["frame_time_delta", "arp_opcode", "icmp_type_code", "tcp_seq", "tcp_hdr_len", "data_len"] + FLAGS
FEATURE_SETS = {
    "header_only": HEADER,
    "with_shortcuts": HEADER + ["frame_number", "arp_src_device", "arp_dst_device"],
}
ROLES = {"attacker": 0, "victim": 1, "unresolved": 2, "other_1": 3, "other_2": 4, "other_3": 5, "other_4": 6}


def encode(df):
    x = df.copy()
    x["arp_opcode"] = x["arp_opcode"].map({"request": 1, "reply": 2})
    for col in x.columns.intersection(["arp_src_device", "arp_dst_device"]):
        x[col] = x[col].map(ROLES)
    return x.astype(float)


rows, preds = [], {}
for name, cols in FEATURE_SETS.items():
    model = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
    model.fit(encode(train[cols]), train["label"])
    preds[name] = model.predict(encode(test[cols]))
    report = classification_report(test["label"], preds[name], target_names=ORDER, output_dict=True)
    rows.append({"features": name, "macro_f1": f1_score(test["label"], preds[name], average="macro"),
                 **{f"recall_{c}": report[c]["recall"] for c in ORDER}})

pd.DataFrame(rows).set_index("features").round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, name in zip(axes, FEATURE_SETS):
    ConfusionMatrixDisplay.from_predictions(test["label"], preds[name], display_labels=ORDER, ax=ax,
                                            xticks_rotation=45, colorbar=False)
    ax.set_title(name)
    ax.grid(False)
plt.tight_layout()
plt.show()

## Findings

1. With `frame_number` and the device roles, the score is about 1.000. The model has learned this capture.
2. With header fields only, the score drops and ARP Storm is the hard class: an ARP request from the attacker looks like any other ARP request.
3. Per-packet classification is close to a lookup here. Rate and window features over `frame_time_delta` (packets per second, share of SYN segments in a window) are the more realistic next step.
4. This is one capture on one network with one attacker. Do not expect the numbers to carry over to other networks.

**Citation**

```
Rakić, A. (2024-2026). Cyber Attack Dataset: ARP, SYN, PING Flood (v3) [Data set]. Kaggle.
```